#### Using python [sqlframe](https://github.com/eakmanrq/sqlframe) library to use PySpark dataframe API against a PostgreSQL database

In [1]:
from datetime import datetime
from dotenv import load_dotenv
import os
from psycopg2 import connect
from sqlframe.postgres import functions as F
from sqlframe.postgres import PostgresSession
from sqlframe.postgres import Window

In [2]:
load_dotenv()

conn = connect(
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=os.environ["DB_PORT"],
)

"""One caveat: autocommit mode means each statement is its own transaction, so if a multi-statement write ever needs to be atomic
(e.g., you want the drop and create to succeed or fail together), you'd lose that guarantee. For a single saveAsTable() call this isn't a concern,
but keep it in mind if your script grows."""
conn.autocommit = True  # ensures DDL/writes are visible to other connections immediately, most suitable for EDA workflow

session = PostgresSession(conn=conn)

#### Reading a PostgreSQL table as a dataframe

In [3]:
df = (
    session.table('public.vehicles')
    .where(F.col("year") == "2027")
    .select("year","make","model","fueltype","fueltype1","fueltype2")
)

In [4]:
df.printSchema()

root
 |-- year: int (nullable = true)
 |-- make: string (nullable = true)
 |-- model: string (nullable = true)
 |-- fueltype: string (nullable = true)
 |-- fueltype1: string (nullable = true)
 |-- fueltype2: string (nullable = true)


sqlframe has features that PySpark does not have like `saveAsTable()` method, which allows you to save a dataframe as a table with just one line of code.  Unfortunately, there is currently a bug where if using PostgreSQL as the backend, the `saveAsTable()` method will fail because the underlying SQL that sqlframe generates is using a dialect using `CREATE OR REPLACE TABLE` syntax which PostgreSQL does not support.

In [5]:
df.write.mode("overwrite").saveAsTable("my_new_table")
conn.commit()

SyntaxError: syntax error at or near "TABLE"
LINE 1: CREATE OR REPLACE TABLE "my_new_table" AS WITH "t30273977" A...
                          ^


**Workaround:** Execute with correct SQL statements using connection cursor

In [6]:
# Drop the target table first (if it exists), since sqlframe's
# mode("overwrite") emits "CREATE OR REPLACE TABLE", which Postgres
# doesn't support.
with conn.cursor() as cur:
    cur.execute('DROP TABLE IF EXISTS public.models_2027')

# No mode() needed now — the table doesn't exist, so this generates
# a plain CREATE TABLE ... AS SELECT
df.write.saveAsTable('public.models_2027')

#### Let's check that our new table was actually made and has data in it

In [7]:
table = session.table("public.models_2027")
table.limit(5).show()

+------+----------+-------------------+----------+------------------+-----------+
| year |   make   |       model       | fueltype |    fueltype1     | fueltype2 |
+------+----------+-------------------+----------+------------------+-----------+
| 2027 | Chrysler |    Pacifica AWD   | Regular  | Regular Gasoline |           |
| 2027 |  Lotus   |       Emira       | Regular  | Regular Gasoline |           |
| 2027 |   BMW    |     430i Coupe    | Premium  | Premium Gasoline |           |
| 2027 |   BMW    | 430i xDrive Coupe | Premium  | Premium Gasoline |           |
| 2027 |   BMW    |  430i Convertible | Premium  | Premium Gasoline |           |
+------+----------+-------------------+----------+------------------+-----------+


#### We can also use `listTables()` to obtain a list of tables

In [8]:
tables = session.catalog.listTables()

In [9]:
type(tables)

list

In [10]:
for table in tables:
    print(table)

Table(name='vehicles', catalog='postgres', namespace=['public'], description=None, tableType='MANAGED', isTemporary=False)
Table(name='models_2027', catalog='postgres', namespace=['public'], description=None, tableType='MANAGED', isTemporary=False)


#### Some filtering examples - dataframe containing 2027 models that are fully electric or partionally electric vehicles (hybrid, plug-in hybrid, etc)

In [11]:
df_elec = (
    session.table('public.vehicles')
    .filter(
        (F.col("fueltype").like("%Elec%")) &
        (F.col("year") == 2027)
    )
    .select("year", "make", "model", "fueltype", "fueltype1", "fueltype2", "startstop", "atvtype")
)

df_elec.limit(5).show()

+------+------+------------------------------------+-------------+-------------+-----------+-----------+---------+
| year | make |               model                |   fueltype  |  fueltype1  | fueltype2 | startstop | atvtype |
+------+------+------------------------------------+-------------+-------------+-----------+-----------+---------+
| 2027 | BMW  | i5 eDrive40 Sedan (19 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 eDrive40 Sedan (20 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 eDrive40 Sedan (21 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 xDrive40 Sedan (20 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 xDrive40 Sedan (21 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
+------+------+------------------------------------+-------------+-------------+

#### Create dataframe containing just ICE, non-hybrid, non-Flexible Fuel vehicles

In [12]:
df_gas = (
    session.table('public.vehicles')
    .filter(
        # Keep only standard combustion engine fuels
        (F.col("fuelType1").isin("Regular Gasoline", "Premium Gasoline", "Midgrade Gasoline")) &
        
        # Exclude Hybrids, PHEVs, and Bi-Fuels (Ensures no secondary fuel source exists)
        (F.col("fuelType2").isNull() | (F.col("fuelType2") == "")) &
        
        # Bulletproof safety net using atvType for any alternative powertrains
        (F.col("atvType").isNull() | (~F.col("atvType").isin("Hybrid", "Plug-in Hybrid", "FFV", "EV", "CNG")))
    )
    .select("year", "make", "model", "fueltype", "fueltype1", "fueltype2", "startstop", "atvtype", "comb08", "highway08")
)

df_gas.limit(5).show()

+------+------------+---------------------+----------+------------------+-----------+-----------+---------+--------+-----------+
| year |    make    |        model        | fueltype |    fueltype1     | fueltype2 | startstop | atvtype | comb08 | highway08 |
+------+------------+---------------------+----------+------------------+-----------+-----------+---------+--------+-----------+
| 1985 | Alfa Romeo |  Spider Veloce 2000 | Regular  | Regular Gasoline |           |           |         |   21   |     25    |
| 1985 |  Ferrari   |      Testarossa     | Regular  | Regular Gasoline |           |           |         |   11   |     14    |
| 1985 |   Dodge    |       Charger       | Regular  | Regular Gasoline |           |           |         |   27   |     33    |
| 1985 |   Dodge    | B150/B250 Wagon 2WD | Regular  | Regular Gasoline |           |           |         |   11   |     12    |
| 1993 |   Subaru   |   Legacy AWD Turbo  | Premium  | Premium Gasoline |           |           |

In [13]:
df_gas.count()

43370

#### ICE, non-diesel, non-hybrid, non-Flexible Fuel vehicles with highest highway08 (highway MPG) from each model year since 2010

In [14]:
window_spec = Window.partitionBy("year").orderBy(F.col("highway08").desc())

current_year = datetime.now().year

df_gas.filter(
    (F.col("year") >= 2010) &
    # Exclude incomplete model year
    (F.col("year") != current_year + 1)
).withColumn(
    "yr_rank", F.dense_rank().over(window_spec)
).filter(
    F.col("yr_rank") <= 1
).select(
    "year",
    "make",
    "model",
    "yr_rank",
    "highway08",
    "fueltype",
    "fueltype1",
    "fueltype2",
    "atvtype"
).orderBy(
    F.col("year").desc(),
    F.col("yr_rank").asc()
).show()

+------+------------+---------------------+---------+-----------+----------+------------------+-----------+---------+
| year |    make    |        model        | yr_rank | highway08 | fueltype |    fueltype1     | fueltype2 | atvtype |
+------+------------+---------------------+---------+-----------+----------+------------------+-----------+---------+
| 2026 |   Toyota   |  Corolla Hatchback  |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2026 |   Honda    |      Civic 4Dr      |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2026 |   Toyota   | Corolla (1-mode TM) |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2025 |   Toyota   |  Corolla Hatchback  |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2025 |  Hyundai   |       Elantra       |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2025 |   Toyota   |       Corolla       |    1    |   

#### ICE, non-diesel, non-hybrid, non-Flexible Fuel vehicles with highest combined MPG (comb08) from each model year since 2010

In [15]:
window_spec = Window.partitionBy("year").orderBy(F.col("comb08").desc())

current_year = datetime.now().year

df_gas.filter(
    (F.col("year") >= 2010) &
    # Exclude incomplete model year
    (F.col("year") != current_year + 1)
).withColumn(
    "yr_rank", F.dense_rank().over(window_spec)
).filter(
    F.col("yr_rank") <= 1
).select(
    "year",
    "make",
    "model",
    "yr_rank",
    "comb08",
    "fueltype",
    "fueltype1",
    "fueltype2",
    "atvtype"
).orderBy(
    F.col("year").desc(),
    F.col("yr_rank").asc()
).show()

+------+------------+------------------+---------+--------+----------+------------------+-----------+---------+
| year |    make    |      model       | yr_rank | comb08 | fueltype |    fueltype1     | fueltype2 | atvtype |
+------+------------+------------------+---------+--------+----------+------------------+-----------+---------+
| 2026 |   Honda    |    Civic 4Dr     |    1    |   36   | Regular  | Regular Gasoline |           |         |
| 2025 |  Hyundai   |     Elantra      |    1    |   36   | Regular  | Regular Gasoline |           |         |
| 2025 |   Honda    |    Civic 4Dr     |    1    |   36   | Regular  | Regular Gasoline |           |         |
| 2024 | Mitsubishi |      Mirage      |    1    |   39   | Regular  | Regular Gasoline |           |         |
| 2023 | Mitsubishi |      Mirage      |    1    |   39   | Regular  | Regular Gasoline |           |         |
| 2022 | Mitsubishi |      Mirage      |    1    |   39   | Regular  | Regular Gasoline |           |   

#### Does combined MPG improve with each subsequent model year?  One would think so, but let's see.

In [16]:
current_year = datetime.now().year

df_gas.filter(
    (F.col("year") >= 2000) &
    # Exclude incomplete new model year
    (F.col("year") != current_year + 1)
).groupBy(
    "year"
).agg(
    F.mean("comb08").alias("avg_comb08")
).select(
    "year",
    "avg_comb08"
).orderBy(
    F.col("year").desc()
).show()

+------+--------------------+
| year |     avg_comb08     |
+------+--------------------+
| 2026 | 22.09660107334526  |
| 2025 | 22.30031948881789  |
| 2024 | 22.09441489361702  |
| 2023 | 22.008705114254624 |
| 2022 | 22.175278622087134 |
| 2021 | 22.275520317145688 |
| 2020 | 22.634099616858236 |
| 2019 | 22.710387323943664 |
| 2018 | 22.825672159583696 |
| 2017 | 22.801075268817204 |
| 2016 | 22.717086834733895 |
| 2015 | 22.413857677902623 |
| 2014 | 22.27469135802469  |
| 2013 | 22.22566844919786  |
| 2012 | 21.407327586206897 |
| 2011 | 20.838219326818674 |
| 2010 | 20.466735966735968 |
| 2009 | 19.738095238095237 |
| 2008 | 19.19178082191781  |
| 2007 | 18.98751200768492  |
+------+--------------------+


Interesting, we peaked at 2018 and slowly declined after that

In [ ]:
current_year = datetime.now().year

df_gas.filter(
    (F.col("year") >= 2000) &
    # Exclude incomplete new model year
    (F.col("year") != current_year + 1)
).groupBy(
    "year"
).agg(
    F.mean("comb08").alias("avg_comb08")
).select(
    "year",
    "avg_comb08"
).orderBy(
    F.col("year").desc()
)

In [17]:
pdf = df_gas.filter(
(
    F.col("year") >= 2000) &
    # Exclude incomplete new model year
    (F.col("year") != current_year + 1)
).groupBy(
    "year"
).agg(
    F.mean("comb08").alias("avg_comb08")
).select(
    "year",
    "avg_comb08"
).orderBy(
    F.col("year").desc()
).limit(10000).toPandas()

C:\Users\danie\envs\pydata_dev\Lib\site-packages\sqlframe\base\session.py:588: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return read_sql_query(


In [18]:
pdf

,year,avg_comb08
0,2026,22.096601
1,2025,22.300319
2,2024,22.094415
3,2023,22.008705
4,2022,22.175279
5,2021,22.275520
6,2020,22.634100
7,2019,22.710387
8,2018,22.825672
9,2017,22.801075
